## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_08_ExG.csv'
MARKER_FILE = 'Case_08_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 2, 3, 9, 10] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz): Asymetria i Sterowanie
Obserwacja kory ruchowej (C3/C4): W tych topomapach nie widać wyraźnej dysocjacji C3 vs C4 na żadnym z wykresów. Cały pas centralny (motoryczny) jest równomiernie wyciszony (niebieski). Dominująca aktywność to wyłącznie kora potyliczna (wzrokowa).

**Faza 1 (Brainrot)**: Bardzo wysoka Alpha z tyłu głowy (ciemna czerwień nad O1, O2, Oz). Mózg w trybie biernego oglądania wchodzi w klasyczny stan "odpoczynku" wzrokowego lub lekkiego transu.

**Fazy ze scrollem** (Brainrot Scroll i Smart Scroll): Fizyczny akt przewijania i rejestrowania nowej treści wyraźnie wycisza fale Alpha (kolor staje się bledszy). Kora wzrokowa "wybudza się" z transu na moment zdekodowania nowego obrazu, niezależnie od tego, jaki to tryb.

#### Analiza Pasma Beta (13–30 Hz): "Efekt Tunelowy"
Obserwacja: Cała aktywność skupia się z tyłu głowy (kora potyliczna). Oznacza to ogromne, ciągłe skupienie uwagi  na bodźcach wizualnych.

**Faza 1 vs Faza 4**: Najsilniejsza aktywacja Beta (ciemnoczerwona plama) występuje w Fazie 1 (Brainrot), to "zwykłe" oglądanie szybkiego przebodźcowującego wideo bez przewijania wymusza na korze wzrokowej najwyższe obroty analityczne.

#### Analiza Pasma Delta (1–4 Hz): Poziomy Świadomości
**Faza 1 (Brainrot)**: Delta jest zerowa (cała głowa granatowa). Mózg wcale nie jest uśpiony.

**Faza 2 (Smart - bez scrolla)**: Pojawia się potężna, ciemnoczerwona plama w okolicach lewej potylicy/ciemienia (O1/P7). Silna, zlokalizowana Delta u zdrowej, przytomnej osoby może oznaczać punktowe wyczerpanie poznawcze, skrajne znużenie analizą "trudnego" materiału lub artefakt od ciągłego czytania tekstu.

**Fazy ze scrollem**: Ruch ręką i zmiana ekranu błyskawicznie "resetują" to zmęczenie.

#### Analiza Pasma Gamma (>30 Hz): Koszt Fizjologiczny
**Brainrot vs Smart**: Gwałtowny wzrost Gammy (>1.2) ma miejsce w Fazie 1 (Brainrot).
Szybkie, zmieniające się jaskrawe obrazy zmuszają korę wzrokową do ekstremalnie szybkiego "sklejania" klatek w jedną całość (tzw. visual binding), co generuje potężny koszt fizjologiczny w paśmie Gamma w rejonach O1/O2.

---

"Pożar" w potylicy: Uczestnik jest silnie przebodźcowany szybkimi filmikami typu Brainrot. Kiedy ich nie przewija (tylko patrzy), jego kora wzrokowa pracuje na skrajnie wysokich obrotach (bardzo wysoka Beta i Gamma), próbując to wszystko zdekodować.

Smart Content go "Znieczula": Zmiana treści na mądrzejszą/tekstową (Smart) obniża tę potężną aktywność Gammy, ale natychmiast wywołuje u niego silne zmęczenie poznawcze lub znudzenie (potężny wystrzał Delty).

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3) – Stabilne, pozytywne zaangażowanie
U tego uczestnika wszystkie słupki są na wyraźnym plusie.
Ta osoba ma bardzo stabilny, pozytywny afekt (motywację dążeniową) do telefonu. Nie ma tu drastycznego znużenia ("odcięcia"), ani ekstremalnej hiper-stymulacji. Przetwarza zarówno głupotki (Brainrot), jak i mądrzejsze treści (Smart) z równym, zdrowym zaciekawieniem i pełną akceptacją. Co ciekawe, w trybie Brainrot zaangażowanie ogólne jest odrobinę wyższe niż w trybie Smart.

#### Kora Ruchowa (C4 vs C3) – Ciągła gotowość motoryczna
Wszystkie wartości są dodatnie, co potwierdza dominację lewej kory ruchowej (obsługa telefonu prawą ręką).
Widać, że w momencie samego aktu scrollowania (Brainrot Scroll i Smart Scroll) słupki rosną pod 0.7 (maksymalna aktywacja do ruchu palcem). Ale nawet podczas "zwykłego" oglądania, wskaźnik jest na poziomie 0.5-0.6. Może to oznaczać, że prawa ręka tego uczestnika jest w ciągłym napięciu – trzyma palec tuż nad ekranem, w pełnej gotowości do natychmiastowego przewinięcia.

#### Kora Ciemieniowa (P4 vs P3) – Skupienie przez działanie
Wartości są umiarkowanie dodatnie. Prawa półkula ciemieniowa (odpowiadająca za stres przestrzenny/przebodźcowanie) jest zrelaksowana.
Uczestnik czuje się w pełni komfortowo. Ciekawy jest fakt, że wskaźnik ten rośnie w fazach scrollowania w porównaniu do biernego oglądania. Oznacza to, że fizyczny akt pociągnięcia palcem po ekranie pomaga mu skupić uwagę na nowym bodźcu.

#### Kora Potyliczna (O2 vs O1)
Wskazuje on na bardzo specyficzny styl przetwarzania wizualnego w zależności od treści:

* Tryb Smart (słupek poniżej zera): Dominacja prawej półkuli wzrokowej. Oznacza to, że treści "Smart" uczestnik przetwarza holistycznie, obrazowo, patrząc na "szerszy obraz". Zamiast skanować tekst słowo po słowie.

* Tryb Brainrot (słupki lekko powyżej zera): Dominacja lewej półkuli wzrokowej. Lewa strona odpowiada za analizę szczegółową, sekwencyjną i dekodowanie języka. Oznacza to, że przy filmikach (Brainrot) mózg przełącza się w tryb "skanera detali" – wyłapuje pojedyncze słowa z napisów, szybkie cięcia i drobne elementy, próbując złożyć z nich sens, klatka po klatce.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_08_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora ruchowa (Beta C3) – "Sprężyna motoryczna"

**Smart Scroll**: Tuż przed przewinięciem linia fali Beta lekko opada (desynchronizacja – układ nerwowy daje sygnał "uwaga, ruszamy kciukiem"), a ułamek sekundy po scrollu występuje tzw. rebound (odbicie) – wyraźny pik w górę. To zdrowy odruch wykonania i zakończenia ruchu.

**Brainrot Scroll**: Mimo że badany przewija często, jego mózg wciąż świetnie izoluje te ruchy.

### Kora wzrokowa (Alpha O1) – "Skaner detali w akcji"

**Smart Scroll**: Bezpośrednio po niebieskiej linii fioletowy wykres fali Alpha natychmiast nurkuje. To moment, kiedy na ekranie pojawia się tekst/zdjęcie, a kora wzrokowa badanego uspokaja się i "chłonie" ten nowy obraz całościowo.

**Brainrot Scroll**: Czerwone linie (scrolle) padają gęsto, a wykres Alfy skacze od zera do wysokich pików. W trybie Brainrot jego mózg przechodzi w tryb "skanera detali" – nerwowo analizuje każdy ułamek sekundy nowego wideo.

### Płaty czołowe (Theta Fz, F4) – "Cykl zrozumienia vs Szok poznawczy"
To pasmo pokazuje obciążenie pamięci roboczej i uwagę.

**Smart Scroll (Fz i F4)**: Tuż za niebieską linią linia Thety zaczyna miarowo rosnąć, tworzy wyraźną "górkę" i opada. To dowód na ułożony proces myślowy: Przewijanie -> Ocenianie nowej treści -> Rozumienie-> Odpuszczanie napięcia.

**Brainrot Scroll (Fz i F4)**: Gigantyczny pik (ponad 10-12 uV) w rejonie 16. sekundy. Występuje dokładnie pomiędzy dwoma szybkimi scrollami. Uczestnik przewinął wideo, a po kilku sekundach jego płat czołowy natrafił na coś stymulującego lub wymagającego analizy, że jego "procesor" poznawczy zaczął silnie pracować, po czym od razu uciął to kolejnym scrollem. Podobny To uderzenia czystej dopaminy i nagłej koncentracji punktowej.

---
W trybie Smart, scroll uruchamia gładki cykl: Ruch -> Spadek Alfy (czytanie) -> Górka Thety (zrozumienie).
W trybie Brainrot uczestnik używa scrolla jako wyzwalacza polowania na "złote strzały" – kora ruchowa uderza jak młot, wzrok nerwowo skanuje klatki wideo, by od czasu do czasu wywołać w płatach czołowych potężną eksplozję uwagi (piki Theta).